In [0]:
import json

# ------------------------------------------------------------------ #
# Load the same config used in ADLS_Databricks_Connection notebook
# ------------------------------------------------------------------ #
CONFIG_PATH = "/Workspace/Users/pavanreddy_adf@outlook.com/Azure_Databricks/etl_config.json"

with open(CONFIG_PATH, "r") as f:
    config = json.load(f)

adls_cfg  = config["adls_connection"]
oauth_cfg = config["oauth_config"]

storage_account = adls_cfg["storage_account"]
container       = adls_cfg["container"]
client_id       = adls_cfg["client_id"]
tenant_id       = adls_cfg["tenant_id"]

# ------------------------------------------------------------------ #
# Configure OAuth (Service Principal - Client Credentials)
# ------------------------------------------------------------------ #
client_secret = dbutils.secrets.get(
    scope=adls_cfg["secret_scope"],
    key=adls_cfg["secret_key"]
)

base     = f"{storage_account}.dfs.core.windows.net"
endpoint = oauth_cfg["endpoint_template"].format(tenant_id=tenant_id)

placeholders = 
{
    "base": base,
    "auth_type": oauth_cfg["auth_type"],
    "provider_type": oauth_cfg["provider_type"],
    "client_id": client_id,
    "client_secret": client_secret,
    "endpoint": endpoint,
}

for key_tmpl, val_tmpl in oauth_cfg["spark_properties"].items():
    spark.conf.set(
        key_tmpl.format(**placeholders),
        val_tmpl.format(**placeholders)
    )

# ------------------------------------------------------------------ #
# Helper: build ABFSS path
# ------------------------------------------------------------------ #
def get_adls_path(path=""):
    base_uri = f"abfss://{container}@{storage_account}.dfs.core.windows.net"
    return f"{base_uri}/{path}" if path else f"{base_uri}/"

print(f"✔ ADLS connection configured for: {storage_account} (container: {container})")
print(f"  Base path: {get_adls_path()}")

In [0]:
# Browse files in the ADLS container to find available CSVs
print("--- Root level ---")
display(dbutils.fs.ls(get_adls_path()))

print("\n--- ainput1/ folder ---")
display(dbutils.fs.ls(get_adls_path("ainput1")))

In [0]:
from pyspark.sql.types import StructType, StructField, IntegerType, StringType, DoubleType

# ------------------------------------------------------------------ #
# Define explicit schema (avoids extra data scan from inferSchema)
# ------------------------------------------------------------------ #
csv_schema = StructType([
    StructField("job_id",                      IntegerType(), True),
    StructField("job_role",                     StringType(),  True),
    StructField("industry",                     StringType(),  True),
    StructField("country",                      StringType(),  True),
    StructField("year",                         IntegerType(), True),
    StructField("automation_risk_percent",       DoubleType(),  True),
    StructField("ai_replacement_score",          DoubleType(),  True),
    StructField("skill_gap_index",               DoubleType(),  True),
    StructField("salary_before_usd",             DoubleType(),  True),
    StructField("salary_after_usd",              DoubleType(),  True),
    StructField("salary_change_percent",          DoubleType(),  True),
    StructField("skill_demand_growth_percent",    DoubleType(),  True),
    StructField("remote_feasibility_score",       DoubleType(),  True),
    StructField("ai_adoption_level",              DoubleType(),  True),
    StructField("education_requirement_level",    IntegerType(), True),
    StructField("automation_risk_category",       StringType(),  True),
    StructField("skill_transition_pressure",      DoubleType(),  True),
    StructField("wage_volatility_index",          DoubleType(),  True),
    StructField("reskilling_urgency_score",       DoubleType(),  True),
    StructField("ai_disruption_intensity",        DoubleType(),  True),
])

# ------------------------------------------------------------------ #
# Load CSV with explicit schema (no inferSchema)
# ------------------------------------------------------------------ #
csv_path = get_adls_path("ainput1/ai_job_replacement_2020_2026_v2.csv")

df = (
    spark.read
    .format("csv")
    .option("header", "true")
    .schema(csv_schema)
    .load(csv_path)
)

print(f"✔ CSV loaded from: {csv_path}")
print(f"  Rows: {df.count():,}, Columns: {len(df.columns)}")
print(f"  Schema:")
df.printSchema()

display(df)

In [0]:
# ------------------------------------------------------------------ #
# Save df as a Delta table in Unity Catalog
# Catalog: Azuredatabricks1 | Schema: impact_ai
# ------------------------------------------------------------------ #

CATALOG = "Azuredatabricks1"
SCHEMA  = "impact_ai"
TABLE   = "ai_job_replacement"

FULL_TABLE_NAME = f"{CATALOG}.{SCHEMA}.{TABLE}"

# Ensure schema exists (catalog already exists in workspace)
spark.sql(f"CREATE SCHEMA IF NOT EXISTS {CATALOG}.{SCHEMA}")

# Write DataFrame as a managed Delta table
df.write.format("delta").mode("overwrite").saveAsTable(FULL_TABLE_NAME)

print(f"✔ Delta table saved: {FULL_TABLE_NAME}")
print(f"  Rows: {spark.table(FULL_TABLE_NAME).count():,}")

# Verify by reading back
display(spark.sql(f"DESCRIBE TABLE EXTENDED {FULL_TABLE_NAME}"))

In [0]:
from pyspark.sql.functions import col, round as spark_round, countDistinct

# ------------------------------------------------------------------ #
# Read from Unity Catalog Delta table
# ------------------------------------------------------------------ #
df_uc = spark.table("Azuredatabricks1.impact_ai.ai_job_replacement")

# ------------------------------------------------------------------ #
# Filter: High AI adoption (>= 70) AND salary decreased after AI
# ------------------------------------------------------------------ #
AI_ADOPTION_THRESHOLD = 70

df_high_ai_salary_drop = (
    df_uc
    .filter(
        (col("ai_adoption_level") >= AI_ADOPTION_THRESHOLD) &
        (col("salary_change_percent") < 0)
    )
    .select(
        "job_id",
        "job_role",
        "industry",
        "country",
        "year",
        spark_round(col("ai_adoption_level"), 2).alias("ai_adoption_level"),
        spark_round(col("salary_before_usd"), 2).alias("salary_before_usd"),
        spark_round(col("salary_after_usd"), 2).alias("salary_after_usd"),
        spark_round(col("salary_change_percent"), 2).alias("salary_change_pct"),
        "automation_risk_category"
    )
    .orderBy(col("salary_change_percent").asc())
)

print(f"Total records matching criteria: {df_high_ai_salary_drop.count():,}")
print(f"Distinct job roles affected: {df_high_ai_salary_drop.select('job_role').distinct().count()}")
print(f"Filter: ai_adoption_level >= {AI_ADOPTION_THRESHOLD} AND salary_change_percent < 0\n")

display(df_high_ai_salary_drop)